## Test crawl4ai

In [ ]:
import asyncio
from crawl4ai import *

async def main():
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url="https://www.nbcnews.com/business",
        )
        print(result.markdown)

if __name__ == "__main__":
    await main()
    

[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://www.nbcnews.com/business                                                                     |
✓ | ⏱: 2.51s 

[SCRAPE].. ◆ https://www.nbcnews.com/business                                                                     |
✓ | ⏱: 0.06s 

[COMPLETE] ● https://www.nbcnews.com/business                                                                     |
✓ | ⏱: 2.58s 

IE 11 is not supported. For an optimal experience visit our site on another browser.
Skip to Content
[NBC News Logo](https://www.nbcnews.com)
Sponsored By
  * [ Politics](https://www.nbcnews.com/politics)
  * [ U.S. News](https://www.nbcnews.com/us-news)
  * [ World](https://www.nbcnews.com/world)
  * Local
    *       * [New York](https://www.nbcnews.com/new-york)
      * [Los Angeles](https://www.nbcnews.com/los-angeles)
      * [Chicago](https://www.nbcnews.com/chicago)
      * [Dallas-Fort Worth](https://www.nbcnews.com/dallas-fort-worth)
      * [Philadelphia](https://www.nbcnews.com/philadelphia)
      * [Washington, D.C.](https://www.nbcnews.com/washington)
      * [Boston](https://www.nbcnews.com/boston)
      * [Bay Area](https://www.nbcnews.com/bay-area)
      * [South Florida](https://www.nbcnews.com/miami)
      * [San Diego](https://www.nbcnews.com/san-diego)
      * [Connecticut](https://www.nbcnews.com/connecticut)
  * [ Sports](http://www.nbcnews.com/sports)
  * [ Healt

# Install the package
pip install -U crawl4ai

# For pre release versions
pip install crawl4ai --pre

# Run post-installation setup
crawl4ai-setup

# Verify your installation
crawl4ai-doctor

In [12]:
async def main(url):
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url,
        )
        return result.markdown
        
url="https://www.reddit.com/r/SingaporeRaw/"
print(await main(url))

[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://www.reddit.com/r/SingaporeRaw/                                                               |
✓ | ⏱: 2.27s 

[SCRAPE].. ◆ https://www.reddit.com/r/SingaporeRaw/                                                               |
✓ | ⏱: 0.03s 

[COMPLETE] ● https://www.reddit.com/r/SingaporeRaw/                                                               |
✓ | ⏱: 2.31s 

[ Skip to main content ](https://www.reddit.com/r/SingaporeRaw/#main-content)
Open menu Open navigation [ ](https://www.reddit.com/)Go to Reddit Home
r/SingaporeRaw
Get App Get the Reddit app  [ Log In ](https://www.reddit.com/login/)Log in to Reddit
Expand user menu Open settings menu
![r/SingaporeRaw icon](https://styles.redditmedia.com/t5_xnx04/styles/communityIcon_niskczlednnd1.jpeg?width=128&frame=1&auto=webp&s=f275bb7fb9a9f4d757f13191e2e8fd708e727670) ![r/SingaporeRaw icon](https://styles.redditmedia.com/t5_xnx04/styles/communityIcon_niskczlednnd1.jpeg?width=128&frame=1&auto=webp&s=f275bb7fb9a9f4d757f13191e2e8fd708e727670)
#  r/SingaporeRaw  ![lionhead](https://emoji.redditmedia.com/d7w1lbkni01f1_t5_xnx04/lionhead) ![lionhead](https://emoji.redditmedia.com/d7w1lbkni01f1_t5_xnx04/lionhead)
SingaporeRaw 
Create Post
[ Feed ](https://www.reddit.com/r/SingaporeRaw) [ About ](https://www.reddit.com/r/SingaporeRaw/about/)
Best
Open sort options 
* [ Best ](https://www.reddit.com/r/Sing

In [5]:
import requests
import pandas as pd
import csv
import time
import datetime
import random

In [ ]:
isubreddit = "WorldCup"
base_url = f"https://www.reddit.com/r/{subreddit}/hot.json"
user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'

# Each request gets ~25 posts. 5 loops = ~125 posts.
loops = 5 

posts = []
after_token = None

print(f"Starting scrape for r/{subreddit}...")

for i in range(loops):
    # Construct the URL with the 'after' token for pagination
    params = {
        'limit': 100,       # Request up to 100 items per call (Reddit max)
        'after': after_token # This tells Reddit where to start the next page
    }
    
    headers = {'User-Agent': user_agent}
    
    try:
        response = requests.get(base_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error: {response.status_code}. You might be rate-limited.")
            break
            
        data = response.json()
        
        # Extract the list of posts
        current_batch = data['data']['children']
        
        # If no posts are returned, we are done
        if not current_batch:
            print("No more posts found.")
            break
            
        for child in current_batch:
            post = child['data']
            posts.append({
                "Title": post.get('title'),
                "Author": post.get('author'),
                "Score": post.get('score'),
                "Comments": post.get('num_comments'),
                "Date": post.get('created_utc'),
                "URL": post.get('url'),
                "Selftext": post.get('selftext') # The body content
            })
            
        # Update the 'after' token for the next loop
        after_token = data['data']['after']
        print(f"Batch {i+1} complete. Total posts: {len(posts)}")
        
        # STOP if there is no next page
        if not after_token:
            break
            
        # IMPORTANT: Wait 2 seconds between requests to avoid being banned
        time.sleep(2)
        
    except Exception as e:
        print(f"An error occurred: {e}")
        break

# 3. Save to CSV
if posts:
    df = pd.DataFrame(posts)
    # Convert timestamp to readable date
    df['Date'] = pd.to_datetime(df['Date'], unit='s')
    
    filename = f"{subreddit}_scraped.csv"
    df.to_csv(filename, index=False)
    print(f"\nSuccess! Saved {len(df)} posts to '{filename}'.")
else:
    print("No posts were scraped.")

Starting scrape for r/WorldCup...
Batch 1 complete. Total posts: 100
Batch 2 complete. Total posts: 200
Batch 3 complete. Total posts: 300
Batch 4 complete. Total posts: 400
Batch 5 complete. Total posts: 500

Success! Saved 500 posts to 'WorldCup_scraped.csv'.


In [ ]:
TARGET_SUBREDDITS = ['soccer', 'WorldCup', 'USMNT', 'CanadaSoccer', 'LigaMX']
SEARCH_QUERY = "World Cup 2026"
OUTPUT_FILE = 'fifa_2026_stealth.csv'

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.3 Safari/605.1.15',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0'
]

def get_json(url, params=None):
    headers = {
        'User-Agent': random.choice(USER_AGENTS),
        'Referer': 'https://www.google.com/'
    }
    try:
        response = session.get(url, headers=headers, params=params, timeout=10)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            print(f"!!! Still Blocked ({response.status_code}). Switching strategy...")
            return None
        else:
            print(f"Error {response.status_code} on {url}")
            return None
    except Exception as e:
        print(f"Exception: {e}")
        return None

def scrape_comments(post_id, subreddit, writer):
    url = f"https://www.reddit.com/r/{subreddit}/comments/{post_id}.json"
    data = get_json(url)
    
    if not data: return

    # Reddit returns a list: [0] is the post, [1] is the comments
    try:
        comments = data[1]['data']['children']
        count = 0
        for comment in comments:
            c_data = comment.get('data', {})
            body = c_data.get('body', '')
            
            # Filter out weak data
            if body and body != '[deleted]' and len(body) > 15:
                writer.writerow([
                    f"r/{subreddit}",
                    c_data.get('id', 'N/A'),
                    datetime.datetime.now().strftime('%Y-%m-%d'), # Approx date
                    'Comment',
                    c_data.get('author', 'unknown'),
                    body.replace('\n', ' '), # Clean newlines
                    c_data.get('score', 0)
                ])
                count += 1
                if count >= 20: break # Limit comments per post to save time
    except (IndexError, KeyError):
        pass

def main():
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['Source', 'ID', 'Date', 'Type', 'Author', 'Text', 'Score'])

        for sub in TARGET_SUBREDDITS:
            print(f"--- Searching r/{sub} ---")
            
            # Search URL endpoint
            base_url = f"https://www.reddit.com/r/{sub}/search.json"
            after_token = None
            
            # Fetch 5 pages of search results per subreddit
            for page in range(5): 
                params = {
                    'q': SEARCH_QUERY,
                    'restrict_sr': 'on',
                    'limit': 25,
                    'sort': 'relevance',
                    'after': after_token
                }
                
                json_data = get_json(base_url, params)
                
                if not json_data: break
                
                posts = json_data.get('data', {}).get('children', [])
                if not posts: break

                for post in posts:
                    p_data = post['data']
                    
                    # 1. Save the Post Title + Body
                    text_content = f"{p_data.get('title', '')} {p_data.get('selftext', '')}"
                    writer.writerow([
                        f"r/{sub}",
                        p_data['id'],
                        datetime.datetime.fromtimestamp(p_data.get('created_utc', 0)),
                        'Post',
                        p_data.get('author', 'unknown'),
                        text_content.replace('\n', ' '),
                        p_data.get('score', 0)
                    ])

                    # 2. Go deeper to get Comments (The "Opinions")
                    # We sleep slightly to avoid hammering the server
                    scrape_comments(p_data['id'], sub, writer)
                    time.sleep(3) 

                # Setup for next page
                after_token = json_data.get('data', {}).get('after')
                print(f"   Page {page+1} done. Next token: {after_token}")
                
                if not after_token: break
                time.sleep(2) # Sleep between search pages

    print(f"Scraping Complete. Data saved to {OUTPUT_FILE}")
main()

--- Searching r/soccer ---
   Page 1 done. Next token: t3_1ocdu8t
   Page 2 done. Next token: t3_1pbijze
   Page 3 done. Next token: t3_1p0p70r
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...
!!! Rate Limit Hit. Sleeping for 30 seconds...


KeyboardInterrupt: 

In [ ]:
import requests
import csv
import time
import random
import os
import signal
import sys

TARGET_SUBREDDITS = ['WorldCup', 'soccer', 'USMNT', 'CanadaSoccer', 'Mexico'] 
OUTPUT_FILE = 'dataset/fifa_2026_marathon.csv'
TARGET_COUNT = 20000  # The Assignment Requirement

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
]

def get_headers():
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.google.com/'
    }

def get_current_count():
    if not os.path.exists(OUTPUT_FILE):
        return 0
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        return sum(1 for line in f) - 1 # Subtract header

def fetch_json(url, params=None):
    retries = 0
    while retries < 5:
        try:
            response = requests.get(url, headers=get_headers(), params=params, timeout=10)
            
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                wait_time = (2 ** retries) * 30  # 30s, 60s, 120s...
                print(f"!!! Rate Limit Hit. Pausing for {wait_time} seconds...")
                time.sleep(wait_time)
                retries += 1
            else:
                print(f"Error {response.status_code}: {url}")
                return None
        except Exception as e:
            print(f"Connection Error: {e}")
            time.sleep(5)
            retries += 1
    return None

def scrape_comments(post_id, subreddit, writer):
    url = f"https://www.reddit.com/r/{subreddit}/comments/{post_id}.json"
    data = fetch_json(url)
    
    if not data: return 0
    
    count = 0
    try:
        # data[1] contains the comments
        comments = data[1]['data']['children']
        for comment in comments:
            c_data = comment.get('data', {})
            body = c_data.get('body', '')
            
            if body and body != '[deleted]' and len(body) > 20:
                writer.writerow([
                    f"r/{subreddit}",
                    c_data.get('id'),
                    'Comment',
                    c_data.get('author'),
                    body.replace('\n', ' '), # Clean for CSV
                    c_data.get('score')
                ])
                count += 1
                if count >= 40: break # Grab max 40 comments per post to stay fast
    except:
        pass
    return count

def main():
    current_count = get_current_count()
    print(f"Current Records: {current_count}")
    print(f"Target: {TARGET_COUNT}")
    
    # Open CSV in Append Mode
    with open(OUTPUT_FILE, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        
        # Write header only if new file
        if current_count == 0:
            writer.writerow(['Source', 'ID', 'Type', 'Author', 'Text', 'Score'])
        
        while current_count < TARGET_COUNT:
            for sub in TARGET_SUBREDDITS:
                print(f"\n>>> Scanning r/{sub} [Records: {current_count}/{TARGET_COUNT}]")
                
                # We use "new" to get the latest chatter, which allows infinite scrolling
                base_url = f"https://www.reddit.com/r/{sub}/new.json"
                after_token = None
                
                # Go 10 pages deep per subreddit pass
                for page in range(10):
                    params = {'limit': 50, 'after': after_token}
                    data = fetch_json(base_url, params)
                    
                    if not data: break
                    
                    posts = data.get('data', {}).get('children', [])
                    if not posts: break
                    
                    for post in posts:
                        p_data = post['data']
                        
                        # Save the Post
                        text = f"{p_data.get('title')} {p_data.get('selftext')}"
                        writer.writerow([
                            f"r/{sub}", 
                            p_data['id'], 
                            'Post', 
                            p_data.get('author'), 
                            text.replace('\n', ' '), 
                            p_data.get('score')
                        ])
                        current_count += 1
                        
                        # Save the Comments (The Volume Booster)
                        # Only check comments for threads with actual discussion (>5 comments)
                        if p_data.get('num_comments', 0) > 5:
                            print(f"   -> Extracting comments from: {p_data['title'][:30]}...")
                            c_added = scrape_comments(p_data['id'], sub, writer)
                            current_count += c_added
                            
                            # Random sleep to mimic reading
                            time.sleep(random.randint(2, 4))
                        
                        if current_count >= TARGET_COUNT:
                            print(f"\n!!! TARGET REACHED: {current_count} RECORDS !!!")
                            sys.exit()

                    after_token = data['data']['after']
                    if not after_token: break
                    
                    # Sleep between pages
                    time.sleep(random.randint(3, 7))

if __name__ == "__main__":
    main()

Current Records: 19518
Target: 20000

>>> Scanning r/WorldCup [Records: 19518/20000]
   -> Extracting comments from: Brazil Chooses New Jersey as I...
   -> Extracting comments from: Is Belgium 2018 one of the gre...
   -> Extracting comments from: Argentina’s national team will...
   -> Extracting comments from: [Culture Shock] To all visitin...
   -> Extracting comments from: Vancouver councillors push for...
   -> Extracting comments from: Messi Confirmed For 2026 World...
   -> Extracting comments from: What do you think about Nigeri...
   -> Extracting comments from: How NFL stadiums are transform...
   -> Extracting comments from: Casual soccer fans: how to get...
   -> Extracting comments from: World Cup Kits… When will kits...
   -> Extracting comments from: Pochettino says World Cup pric...
   -> Extracting comments from: FOX Sports Unveils World Cup 2...
   -> Extracting comments from: The best 48 team FIFA world cu...
   -> Extracting comments from: Leon Goretzka says Donald

SystemExit: 

/Users/leonardong/Documents/School/y4s2/SC4021/SC4021-project/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
